<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/HEADSWAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 1. Install dependencies
!pip install -q diffusers transformers accelerate peft safetensors pillow opencv-python-headless

from IPython.display import clear_output
clear_output()
print("✅ Installed")

✅ Installed


In [ ]:
#@title 2. Load Qwen-Image + BFS Head Swap LoRA
import torch
from diffusers import DiffusionPipeline
from diffusers.utils import load_image
from huggingface_hub import hf_hub_download
import gc

# Clear memory
torch.cuda.empty_cache()
gc.collect()

print("Loading base model (this takes 1-2 minutes)...")

# Using a lighter configuration that works better on T4
pipe = DiffusionPipeline.from_pretrained(
    "Qwen/Qwen-Image-2.1",
    torch_dtype=torch.bfloat16,
    device_map="balanced"          # important for T4
)

print("Loading BFS Head Swap LoRA...")
pipe.load_lora_weights(
    "Alissonerdx/BFS-Best-Face-Swap",
    weight_name="bfs_head_v1.1_qwen_2.1.safetensors"   # recommended version
)

pipe.enable_model_cpu_offload()   # helps a lot on T4
pipe.enable_attention_slicing()

print("✅ Model + LoRA loaded")

Loading base model (this takes 1-2 minutes)...


/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


model_index.json:   0%|          | 0.00/447 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 23 files:   0%|          | 0/23 [00:00<?, ?it/s]

In [ ]:
#@title 3. Upload Body + Head and Run Swap
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt

print("👉 Upload BODY / TARGET image (the person whose body + pose you want to keep)")
body_upload = files.upload()
body_path = list(body_upload.keys())[0]

print("\n👉 Upload HEAD / SOURCE image (the face + head you want to put on the body)")
head_upload = files.upload()
head_path = list(head_upload.keys())[0]

body_img = Image.open(body_path).convert("RGB")
head_img = Image.open(head_path).convert("RGB")

# Recommended prompt for BFS Head Swap
prompt = """head_swap: start with <image1> as the base image, keeping its lighting, environment, and background.
remove the head from <image1> completely and replace it with the head from <image2>,
strictly preserving the hair, eye color, nose structure from <image2>.
copy the direction of the eye, head rotation, micro expressions from <image1>,
high quality, sharp details, 4k"""

print("\nRunning head swap... (this can take 30-90 seconds on T4)")

result = pipe(
    prompt=prompt,
    image=[body_img, head_img],   # order is important: body first, head second
    num_inference_steps=12,
    guidance_scale=1.0,
    height=1024,
    width=1024
).images[0]

result.save("/content/head_swap_result.jpg")
print("✅ Done!")

# Show result
plt.figure(figsize=(12, 6))
plt.subplot(1, 3, 1)
plt.imshow(body_img)
plt.title("Body")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(head_img)
plt.title("Head")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(result)
plt.title("Result")
plt.axis("off")
plt.show()

In [ ]:
#@title 4. Download result
from google.colab import files
files.download("/content/head_swap_result.jpg")